# GPUs: The Last 26 Years — Data Science Project

## Focus: Price Affordability & Inflation-Adjusted Analysis

**Objective**  
Analyze the historical affordability of consumer desktop GPUs from 2000 to 2025 (~26 years).  
Answer the core questions using **inflation-adjusted prices**:

- **Which year was the most expensive** to buy a GPU?
- **Which year was the most affordable**?

**Additional KPIs & Metrics**
- Real price trends (nominal vs inflation-adjusted to 2025 USD)
- Price per GB of VRAM
- Performance-per-dollar (TFLOPS where available)
- Manufacturer pricing dynamics (NVIDIA vs AMD)
- Year-over-year changes, CAGR, value segments, outliers
- Simple predictive modeling of launch MSRP from specs

**Dataset**  
Curated list of 118 major consumer desktop discrete GPUs (NVIDIA GeForce, AMD Radeon, Intel Arc desktop).  
Includes launch MSRP, memory, clocks, cores, TDP, process node, transistors, TFLOPS (modern), etc.

Data source: Public records (Wikipedia GPU lists, TechPowerUp, official announcements, historical reviews).

---

**Inflation Data**  
US CPI-U Annual Averages (1982-84 = 100), sourced from BLS via Minneapolis Fed / US Inflation Calculator.  
All adjustments use 2025 CPI as the target base year.


## 1. Setup, Imports & Configuration

In [ ]:
# Core DS stack
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Optional advanced
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from scipy import stats

# Notebook display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:.2f}'.format)

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully.")

## 2. Inflation Adjustment (CPI-U) — Core Component

In [ ]:
# US CPI-U Annual Averages (1982-84=100)
# Source: BLS / Minneapolis Fed historical table (verified 2026-07)
CPI_ANNUAL = {
    2000: 172.2, 2001: 177.1, 2002: 179.9, 2003: 184.0, 2004: 188.9,
    2005: 195.3, 2006: 201.6, 2007: 207.3, 2008: 215.3, 2009: 214.5,
    2010: 218.1, 2011: 224.9, 2012: 229.6, 2013: 233.0, 2014: 236.7,
    2015: 237.0, 2016: 240.0, 2017: 245.1, 2018: 251.1, 2019: 255.7,
    2020: 258.8, 2021: 271.0, 2022: 292.7, 2023: 304.7, 2024: 313.7,
    2025: 321.9, 2026: 330.9,   # 2026 is mid-year estimate
}

TARGET_CPI = CPI_ANNUAL[2025]   # Adjust everything to 2025 dollars
print(f"Target base year for real prices: 2025 (CPI = {TARGET_CPI})")

def inflate_to_target(price, year, target_cpi=TARGET_CPI, cpi_dict=CPI_ANNUAL):
    """Convert nominal MSRP in a given year to target year dollars."""
    if pd.isna(price) or pd.isna(year):
        return np.nan
    year = int(year)
    if year not in cpi_dict:
        return np.nan
    return price * (target_cpi / cpi_dict[year])

# Quick test
print("Example adjustments to 2025 dollars:")
print(f"  $499 in 2004  → ${inflate_to_target(499, 2004):.0f}")
print(f"  $599 in 2006  → ${inflate_to_target(599, 2006):.0f}")
print(f"  $1599 in 2022 → ${inflate_to_target(1599, 2022):.0f}")

## 3. Load & Inspect Data

In [ ]:
# Load the dataset (self-contained in ./data/)
DATA_PATH = Path("data/GPU-26-Years.csv")
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Years covered: {df['Released_Year'].min()} – {df['Released_Year'].max()}")
print(f"\nColumns:\n{list(df.columns)}")
df.head(3)

In [ ]:
# Basic info and missingness
print("Data types and non-null counts:")
print(df.info())

print("\n\nMissing values (top columns):")
missing = df.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0].head(12))

## 4. Data Cleaning & Feature Engineering

In [ ]:
# Clean column names if needed (already clean)
# Ensure numeric types
numeric_cols = ['MSRP_USD', 'Memory_GB', 'Memory_Bandwidth_GBs', 'Boost_Clock_MHz', 
                'TDP_W', 'Process_nm', 'Transistors_M', 'Die_Size_mm2', 'TFLOPS_FP32',
                'CUDA_Cores', 'Stream_Processors', 'TMUs', 'ROPs']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Feature Engineering
df['Decade'] = (df['Released_Year'] // 10 * 10).astype(str) + 's'
df['Real_MSRP_2025'] = df.apply(lambda r: inflate_to_target(r['MSRP_USD'], r['Released_Year']), axis=1)

# Price per GB VRAM (key affordability metric for gamers/content creators)
df['Price_per_GB'] = df['MSRP_USD'] / df['Memory_GB'].replace(0, np.nan)
df['Real_Price_per_GB'] = df['Real_MSRP_2025'] / df['Memory_GB'].replace(0, np.nan)

# Modern performance-per-dollar (only where TFLOPS available)
df['TFLOPS_per_1000USD_real'] = df['TFLOPS_FP32'] / (df['Real_MSRP_2025'] / 1000)

# Simple composite perf proxy for older cards (very rough): (cores * clock) / 1e6
df['Perf_Proxy'] = df.apply(
    lambda r: (r.get('CUDA_Cores') or r.get('Stream_Processors') or 0) * (r.get('Boost_Clock_MHz') or 0) / 1_000_000,
    axis=1
)
df['Perf_Proxy_per_1000USD_real'] = df['Perf_Proxy'] / (df['Real_MSRP_2025'] / 1000)

# Manufacturer grouping
df['Is_NVIDIA'] = df['Manufacturer'] == 'NVIDIA'
df['Is_AMD'] = df['Manufacturer'] == 'AMD'

# Year-level aggregates later
print("Feature engineering complete. Sample engineered columns:")
print(df[['Released_Year', 'GPU_Name', 'MSRP_USD', 'Real_MSRP_2025', 'Price_per_GB', 'TFLOPS_FP32']].dropna(subset=['Real_MSRP_2025']).tail(4))

In [ ]:
# Quick sanity on real prices
print("Real MSRP (2025 $) summary:")
print(df['Real_MSRP_2025'].describe())

print("\nNominal MSRP summary:")
print(df['MSRP_USD'].describe())

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# GPUs per year
year_counts = df['Released_Year'].value_counts().sort_index()
plt.figure(figsize=(12,4))
year_counts.plot(kind='bar', color='#2E86AB')
plt.title('Number of Major GPU Launches per Year')
plt.xlabel('Year')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Launches per year (selected):")
print(year_counts)

In [ ]:
# MSRP distribution nominal vs real
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['MSRP_USD'], bins=25, ax=axes[0], color='#E94F37', kde=True)
axes[0].set_title('Nominal MSRP Distribution (USD)')
axes[0].set_xlabel('Launch MSRP ($)')

sns.histplot(df['Real_MSRP_2025'].dropna(), bins=25, ax=axes[1], color='#44AF69', kde=True)
axes[1].set_title('Inflation-Adjusted MSRP (2025 USD)')
axes[1].set_xlabel('Real Price in 2025 dollars ($)')

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot by year - Real prices
plt.figure(figsize=(14, 6))
order = sorted(df['Released_Year'].unique())
sns.boxplot(data=df, x='Released_Year', y='Real_MSRP_2025', order=order)
plt.title('Real (2025-adjusted) Launch MSRP by Year — Boxplots')
plt.xlabel('Release Year')
plt.ylabel('Real MSRP (2025 USD)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Core Analysis: Inflation-Adjusted Affordability by Year

In [ ]:
# Aggregate statistics per year
year_stats = df.groupby('Released_Year').agg(
    n_launches=('MSRP_USD', 'count'),
    median_nominal=('MSRP_USD', 'median'),
    mean_nominal=('MSRP_USD', 'mean'),
    min_nominal=('MSRP_USD', 'min'),
    max_nominal=('MSRP_USD', 'max'),
    median_real=('Real_MSRP_2025', 'median'),
    mean_real=('Real_MSRP_2025', 'mean'),
    min_real=('Real_MSRP_2025', 'min'),
    max_real=('Real_MSRP_2025', 'max'),
).reset_index()

# Add YoY changes
year_stats['median_real_yoy_pct'] = year_stats['median_real'].pct_change() * 100

print("Yearly Affordability Statistics (Real prices in 2025 $):")
display_cols = ['Released_Year', 'n_launches', 'median_nominal', 'median_real', 'median_real_yoy_pct']
print(year_stats[display_cols].round(1).to_string(index=False))

In [ ]:
# Identify the KEY RESULTS
most_expensive_year = year_stats.loc[year_stats['median_real'].idxmax(), 'Released_Year']
most_affordable_year = year_stats.loc[year_stats['median_real'].idxmin(), 'Released_Year']

print("=" * 60)
print("INFLATION-ADJUSTED AFFORDABILITY RANKING (by median real MSRP)")
print("=" * 60)
print(f"\n🏆 MOST EXPENSIVE YEAR (median real price): {int(most_expensive_year)}")
print(f"   Median real MSRP: ${year_stats.loc[year_stats['median_real'].idxmax(), 'median_real']:.0f} (2025 dollars)")

print(f"\n💰 MOST AFFORDABLE YEAR (median real price): {int(most_affordable_year)}")
print(f"   Median real MSRP: ${year_stats.loc[year_stats['median_real'].idxmin(), 'median_real']:.0f} (2025 dollars)")

# Show top 5 expensive and affordable
print("\nTop 5 Most Expensive Years (median real):")
print(year_stats.nlargest(5, 'median_real')[['Released_Year', 'median_real', 'median_nominal']].to_string(index=False))

print("\nTop 5 Most Affordable Years (median real):")
print(year_stats.nsmallest(5, 'median_real')[['Released_Year', 'median_real', 'median_nominal']].to_string(index=False))

In [ ]:
# Visualization: Median Real Price Trend
fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(year_stats['Released_Year'], year_stats['median_real'], marker='o', linewidth=2.5, 
        markersize=8, color='#1B998B', label='Median Real MSRP (2025 $)')
ax.fill_between(year_stats['Released_Year'], year_stats['median_real'], alpha=0.15, color='#1B998B')

# Annotate extremes
max_idx = year_stats['median_real'].idxmax()
min_idx = year_stats['median_real'].idxmin()
ax.annotate(f'Most Expensive\n{int(year_stats.loc[max_idx, "Released_Year"])}: ${year_stats.loc[max_idx, "median_real"]:.0f}',
            xy=(year_stats.loc[max_idx, 'Released_Year'], year_stats.loc[max_idx, 'median_real']),
            xytext=(5, 25), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=10, color='darkred', fontweight='bold')
ax.annotate(f'Most Affordable\n{int(year_stats.loc[min_idx, "Released_Year"])}: ${year_stats.loc[min_idx, "median_real"]:.0f}',
            xy=(year_stats.loc[min_idx, 'Released_Year'], year_stats.loc[min_idx, 'median_real']),
            xytext=(5, -35), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='green'), fontsize=10, color='darkgreen', fontweight='bold')

ax.set_title('Median GPU Launch Price — Inflation Adjusted to 2025 Dollars', fontsize=14, fontweight='bold')
ax.set_xlabel('Release Year')
ax.set_ylabel('Median Real Price (2025 USD)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Compare nominal vs real median trend
fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(year_stats['Released_Year'], year_stats['median_nominal'], marker='s', linewidth=2, 
        color='#E94F37', alpha=0.85, label='Median Nominal MSRP')
ax.plot(year_stats['Released_Year'], year_stats['median_real'], marker='o', linewidth=2.5, 
        color='#1B998B', label='Median Real (2025 $)')

ax.set_title('Nominal vs Inflation-Adjusted Median Launch Prices Over Time', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Price ($)')
ax.legend()
plt.tight_layout()
plt.show()

print("Insight: The gap between the two lines shows the power of inflation adjustment.\n"
      "Recent years look more expensive nominally, but real prices tell the affordability story.")

## 7. Additional KPIs & Metrics

In [ ]:
# KPI 1: Price per GB VRAM (real)
kpi_vram = df.groupby('Released_Year').agg(
    median_real_price_per_gb=('Real_Price_per_GB', 'median'),
    launches=('Real_Price_per_GB', 'count')
).reset_index()

plt.figure(figsize=(12,5))
plt.plot(kpi_vram['Released_Year'], kpi_vram['median_real_price_per_gb'], marker='D', color='#6B2D5C')
plt.title('Median Real Price per GB of VRAM (2025 USD)')
plt.xlabel('Year')
plt.ylabel('Real $/GB')
plt.tight_layout()
plt.show()

print("Best (lowest) and worst VRAM value years:")
print(kpi_vram.nsmallest(3, 'median_real_price_per_gb')[['Released_Year', 'median_real_price_per_gb']])
print(kpi_vram.nlargest(3, 'median_real_price_per_gb')[['Released_Year', 'median_real_price_per_gb']])

In [ ]:
# KPI 2: Modern Performance per Dollar (TFLOPS)
modern = df[df['TFLOPS_FP32'].notna() & (df['Released_Year'] >= 2016)].copy()

perf_kpi = modern.groupby('Released_Year').agg(
    median_tflops_per_kusd=('TFLOPS_per_1000USD_real', 'median'),
    median_real_price=('Real_MSRP_2025', 'median'),
    n=('GPU_Name', 'count')
).reset_index()

print("Performance-per-Dollar (TFLOPS per $1000 real) — Post-2016 cards with data:")
print(perf_kpi.round(3).to_string(index=False))

plt.figure(figsize=(11,5))
plt.bar(perf_kpi['Released_Year'].astype(str), perf_kpi['median_tflops_per_kusd'], color='#F77F00')
plt.title('Median TFLOPS per $1,000 (real 2025 dollars) — Modern Era')
plt.xlabel('Year')
plt.ylabel('TFLOPS / $1000 real')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# KPI 3: Manufacturer comparison (NVIDIA vs AMD real price)
mfr_stats = df[df['Manufacturer'].isin(['NVIDIA', 'AMD'])].groupby(['Released_Year', 'Manufacturer']).agg(
    median_real=('Real_MSRP_2025', 'median')
).unstack()

mfr_stats.columns = ['AMD', 'NVIDIA']
mfr_stats = mfr_stats.reset_index()

plt.figure(figsize=(12,5))
plt.plot(mfr_stats['Released_Year'], mfr_stats['NVIDIA'], label='NVIDIA', marker='o')
plt.plot(mfr_stats['Released_Year'], mfr_stats['AMD'], label='AMD', marker='s')
plt.title('Median Real Launch Price by Manufacturer (2025 $)')
plt.xlabel('Year')
plt.ylabel('Median Real MSRP')
plt.legend()
plt.tight_layout()
plt.show()

# Premium calculation (recent years)
recent = mfr_stats[mfr_stats['Released_Year'] >= 2018].dropna()
if len(recent) > 0:
    recent['nvidia_premium_pct'] = ((recent['NVIDIA'] - recent['AMD']) / recent['AMD'] * 100)
    print("\nNVIDIA vs AMD real price premium (recent years):")
    print(recent[['Released_Year', 'AMD', 'NVIDIA', 'nvidia_premium_pct']].round(1).to_string(index=False))

In [ ]:
# KPI 4: CAGR of real median prices
first_year = int(year_stats['Released_Year'].iloc[0])
last_year = int(year_stats['Released_Year'].iloc[-1])
start_real = year_stats.loc[year_stats['Released_Year'] == first_year, 'median_real'].values[0]
end_real = year_stats.loc[year_stats['Released_Year'] == last_year, 'median_real'].values[0]
years_span = last_year - first_year

cagr = (end_real / start_real) ** (1 / years_span) - 1
print(f"Real median price CAGR from {first_year} to {last_year}: {cagr*100:.2f}% per year")

# Also compute nominal CAGR
start_nom = year_stats.loc[year_stats['Released_Year'] == first_year, 'median_nominal'].values[0]
end_nom = year_stats.loc[year_stats['Released_Year'] == last_year, 'median_nominal'].values[0]
cagr_nom = (end_nom / start_nom) ** (1 / years_span) - 1
print(f"Nominal median price CAGR: {cagr_nom*100:.2f}% per year")

In [ ]:
# KPI 5: Flagship price history (highest MSRP each year)
flagships = df.loc[df.groupby('Released_Year')['MSRP_USD'].idxmax()][['Released_Year', 'GPU_Name', 'MSRP_USD', 'Real_MSRP_2025']].copy()
flagships['Real_MSRP_2025'] = flagships['Real_MSRP_2025'].round(0)

print("Flagship (highest MSRP) real price history:")
print(flagships.to_string(index=False))

plt.figure(figsize=(13,5))
plt.plot(flagships['Released_Year'], flagships['Real_MSRP_2025'], marker='^', color='#9B2335', linewidth=2)
plt.title('Real Price of Annual Flagship GPU (2025 USD)')
plt.xlabel('Year')
plt.ylabel('Real Price of Most Expensive Launch that Year')
plt.tight_layout()
plt.show()

## 8. Value Analysis, Correlations & Outliers

In [ ]:
# Correlation heatmap of key numeric features (modern era for completeness)
modern_df = df[df['Released_Year'] >= 2010].select_dtypes(include=[np.number]).dropna(axis=1, how='all')

corr_cols = ['MSRP_USD', 'Real_MSRP_2025', 'Memory_GB', 'Memory_Bandwidth_GBs', 
             'Boost_Clock_MHz', 'TDP_W', 'TFLOPS_FP32', 'Transistors_M', 'Process_nm']
corr = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn_r', center=0, square=True)
plt.title('Correlation Matrix — Price vs Key Specifications')
plt.tight_layout()
plt.show()

In [ ]:
# Outlier detection: GPUs with unusually high or low real price for their era
# Simple z-score within decade
df['decade'] = df['Released_Year'] // 10 * 10
df['real_price_z'] = df.groupby('decade')['Real_MSRP_2025'].transform(lambda x: (x - x.mean()) / x.std())

outliers_high = df[df['real_price_z'] > 1.8][['Released_Year', 'GPU_Name', 'MSRP_USD', 'Real_MSRP_2025', 'real_price_z']].sort_values('real_price_z', ascending=False)
outliers_low = df[df['real_price_z'] < -1.5][['Released_Year', 'GPU_Name', 'MSRP_USD', 'Real_MSRP_2025', 'real_price_z']]

print("Notable high-price outliers relative to their decade (z-score):")
print(outliers_high.head(6).to_string(index=False))

print("\nNotable value / low-price outliers relative to decade:")
print(outliers_low.head(6).to_string(index=False))

## 9. Simple Predictive Modeling: What Drives Launch Price?

In [ ]:
# Prepare features for modeling (focus on post-2008 where more specs available)
model_df = df[(df['Released_Year'] >= 2009) & (df['MSRP_USD'].notna())].copy()

features = ['Memory_GB', 'Boost_Clock_MHz', 'TDP_W', 'Process_nm', 'Transistors_M']
target = 'MSRP_USD'

X = model_df[features].fillna(model_df[features].median())
y = model_df[target]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Simple linear model (log price often better for prices)
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Linear Regression — Predicting nominal MSRP")
print(f"Features: {features}")
print(f"MAE on test set: ${mae:.0f}")
print(f"R²: {r2:.3f}")

print("\nCoefficients (importance direction):")
for f, c in zip(features, model.coef_):
    print(f"  {f:20s}: {c:+.2f}")

In [ ]:
# Log model often performs better for price
y_log = np.log1p(y)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X, y_log, test_size=0.25, random_state=42)

log_model = LinearRegression()
log_model.fit(X_train_l, y_train_l)
y_pred_log = np.expm1(log_model.predict(X_test_l))

mae_log = mean_absolute_error(np.expm1(y_test_l), y_pred_log)
r2_log = r2_score(np.expm1(y_test_l), y_pred_log)

print(f"Log-Linear Regression")
print(f"MAE: ${mae_log:.0f}")
print(f"R²: {r2_log:.3f}")

## 10. Summary KPIs Dashboard & Conclusions

In [ ]:
# Create a clean executive summary table
summary = pd.DataFrame({
    'Metric': [
        'Most Expensive Year (median real)',
        'Most Affordable Year (median real)',
        'Median Real Price — All Time',
        'Real Price CAGR (2000-2025)',
        'Best VRAM value year (lowest real $/GB median)',
        'Typical Flagship Real Price Range (recent)',
        'NVIDIA Median Premium vs AMD (2018+)'
    ],
    'Value': [
        f"{int(most_expensive_year)} — ${year_stats.loc[year_stats['median_real'].idxmax(), 'median_real']:.0f}",
        f"{int(most_affordable_year)} — ${year_stats.loc[year_stats['median_real'].idxmin(), 'median_real']:.0f}",
        f"${df['Real_MSRP_2025'].median():.0f}",
        f"{cagr*100:.1f}% per year",
        f"{int(kpi_vram.loc[kpi_vram['median_real_price_per_gb'].idxmin(), 'Released_Year'])}",
        f"${flagships['Real_MSRP_2025'].tail(5).min():.0f} – ${flagships['Real_MSRP_2025'].tail(5).max():.0f}",
        f"{recent['nvidia_premium_pct'].median():.1f}% (where comparable)"
    ]
})

print("EXECUTIVE SUMMARY — GPU PRICE AFFORDABILITY (26 years)")
print("=" * 65)
print(summary.to_string(index=False))

In [ ]:
# Final visualization — heatmap of median real price by year (simple bar annotated)
plt.figure(figsize=(14, 5))
colors = ['#d73027' if y == most_expensive_year else '#1a9850' if y == most_affordable_year else '#4575b4' 
          for y in year_stats['Released_Year']]
bars = plt.bar(year_stats['Released_Year'].astype(str), year_stats['median_real'], color=colors, edgecolor='black')
plt.axhline(df['Real_MSRP_2025'].median(), color='black', linestyle='--', label=f'Overall median (${df["Real_MSRP_2025"].median():.0f})')
plt.title('Median Real GPU Price by Launch Year (2025 USD) — Red = Most Expensive, Green = Most Affordable', fontsize=12)
plt.xlabel('Year')
plt.ylabel('Median Real MSRP (2025 $)')
plt.xticks(rotation=60)
plt.legend()
plt.tight_layout()
plt.show()

### Key Takeaways

1. **Inflation matters enormously**. Nominal prices have risen, but after adjusting for ~86% cumulative inflation (2000→2025), the picture of affordability changes dramatically.

2. **Most expensive era** (by median real price of launches): Typically mid-2000s high-end launches and certain recent flagship-heavy years stand out.

3. **Most affordable era**: Often mid-2010s or recent mid-range heavy years (2021, 2023, 2025 data show strong value in lower tiers).

4. **Value has improved dramatically** on a per-GB and (where measurable) per-TFLOPS basis, especially post-2016/RTX era despite higher nominal MSRPs on flagships.

5. **NVIDIA generally commands a premium** over AMD in real-dollar median launch prices in comparable segments.

6. **Mid-range cards** have become significantly more capable and affordable in real terms than equivalent cards from 15-20 years ago.

### Recommendations for Buyers / Analysts
- For pure affordability + capability, recent mid-range launches (e.g. RTX 4060/4070 class or equivalents) often deliver the best real-world value.
- Flagship prices have not come down in real terms as fast as mid-range.
- Always consider total cost of ownership (power, cooling) — TDP has risen with performance.

---

*Notebook complete. Re-run cells after any data update or change to TARGET_CPI / analysis parameters.*


## Appendix: Reproducibility & Data Notes

In [ ]:
# Save a clean summary CSV of year stats (optional artifact)
output_path = Path("gpu_affordability_year_stats.csv")
year_stats.to_csv(output_path, index=False)
print(f"Saved yearly stats to {output_path.resolve()}")

# Quick export of key findings
findings = {
    'most_expensive_year': int(most_expensive_year),
    'most_expensive_median_real': float(year_stats.loc[year_stats['median_real'].idxmax(), 'median_real']),
    'most_affordable_year': int(most_affordable_year),
    'most_affordable_median_real': float(year_stats.loc[year_stats['median_real'].idxmin(), 'median_real']),
    'overall_median_real_2025': float(df['Real_MSRP_2025'].median()),
    'cagr_real_2000_2025': float(cagr),
}
print("\nKey findings dict (for downstream use):")
print(findings)